In [ ]:
import os


# 1. مفتاحك السري (صحيح ويعمل 100%)
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f50c0bfce6b4eb0c9e0e70c440728a26"

# 2. تحديث مكتبة Kaggle لتجنب التحذير الأصفر الذي ظهر لك
!pip install --upgrade kaggle -q

# 3. تحميل حزمة البيانات المعتمدة والضخمة (تحتوي على فيديوهات حقيقية ومزيفة)
!kaggle datasets download -d sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset

# 4. فك الضغط داخل مجلد اسمه dataset
# (هذه الحزمة احترافية وحجمها كبير، قد يأخذ فك الضغط بضع دقائق، دعه يعمل براحته)
!unzip -q deep-fake-detection-dfd-entire-original-dataset.zip -d dataset
print("✅ تم تحميل وفك ضغط الفيديوهات الحقيقية والمزيفة بنجاح!")

Dataset URL: https://www.kaggle.com/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset
License(s): MIT
100% 22.5G/22.5G [03:51<00:00, 104MB/s]

✅ تم تحميل وفك ضغط الفيديوهات الحقيقية والمزيفة بنجاح!


In [ ]:
import os

# المسار الذي فككنا فيه الضغط
dataset_path = '/content/dataset'

print("📂 جاري استكشاف محتويات المجلد الرئيسي...")
items = os.listdir(dataset_path)

for item in items:
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        # إذا كان المجلد فرعياً، سنعد الملفات بداخله
        num_files = len(os.listdir(item_path))
        print(f"📁 مجلد: {item} (يحتوي على {num_files} ملف)")
    else:
        print(f"📄 ملف: {item}")


📂 جاري استكشاف محتويات المجلد الرئيسي...
📁 مجلد: DFD_manipulated_sequences (يحتوي على 1 ملف)
📁 مجلد: DFD_original sequences (يحتوي على 364 ملف)


In [ ]:
import os
fake_folder = '/content/dataset/DFD_manipulated_sequences'
محتويات = os.listdir(fake_folder)

print(f"العنصر الوحيد الموجود بالداخل هو: {محتويات[0]}")

# التحقق إذا كان هذا العنصر مجلداً أم ملفاً
مسار_العنصر = os.path.join(fake_folder, محتويات[0])
if os.path.isdir(مسار_العنصر):
    print(f"هذا العنصر عبارة عن 'مجلد'، ويحتوي بداخله على: {len(os.listdir(مسار_العنصر))} فيديو!")
else:
    print("هذا العنصر عبارة عن ملف (ربما ملف مضغوط zip).")

العنصر الوحيد الموجود بالداخل هو: DFD_manipulated_sequences
هذا العنصر عبارة عن 'مجلد'، ويحتوي بداخله على: 3068 فيديو!


In [ ]:
!pip install mtcnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.8 MB/s eta 0:00:00


In [ ]:
!pip install lz4

In [ ]:
import cv2
from mtcnn import MTCNN
import os

# 1. إعداد مسارات الفيديوهات
real_videos_path = '/content/dataset/DFD_original sequences'
fake_videos_path = '/content/dataset/DFD_manipulated_sequences/DFD_manipulated_sequences'

output_real = '/content/extracted_faces/real'
output_fake = '/content/extracted_faces/fake'
os.makedirs(output_real, exist_ok=True)
os.makedirs(output_fake, exist_ok=True)

detector = MTCNN()

def process_sample_videos(video_folder, output_folder, max_videos=200, sequence_length=10):
    videos = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
    videos = videos[:max_videos]

    print(f"🔄 جاري معالجة {len(videos)} فيديوهات من المجلد...")

    for video_name in videos:
        video_path = os.path.join(video_folder, video_name)
        vid_id = video_name.split('.')[0]

        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        saved_count = 0

        while cap.isOpened() and saved_count < sequence_length:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % 15 == 0:
                # --- 🛡️ الدرع الواقي ---
                try:
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    results = detector.detect_faces(rgb_frame)

                    if results:
                        x, y, w, h = results[0]['box']
                        x, y = max(0, x), max(0, y)
                        face = frame[y:y+h, x:x+w]

                        face_resized = cv2.resize(face, (224, 224))
                        filename = os.path.join(output_folder, f"{vid_id}_face_{saved_count}.jpg")
                        cv2.imwrite(filename, cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
                        saved_count += 1
                except Exception as e:
                    # إذا واجه إطاراً معطوباً، سيتجاهله بصمت ويكمل
                    pass
                # -----------------------
            frame_count += 1
        cap.release()
    print(f"✅ تم الانتهاء من استخراج الوجوه وحفظها في: {output_folder}")

# --- الانطلاق نحو تدريب الوحش ---
print("🚀 بدء استخراج عينة الوجوه الحقيقية (200 فيديو)...")
process_sample_videos(real_videos_path, output_real, max_videos=200)

print("🚀 بدء استخراج عينة الوجوه المزيفة (200 فيديو)...")
process_sample_videos(fake_videos_path, output_fake, max_videos=200)

print("🎉 اكتمل تجهيز البيانات الضخمة بنجاح!")

🚀 بدء استخراج عينة الوجوه الحقيقية (200 فيديو)...
🔄 جاري معالجة 200 فيديوهات من المجلد...
✅ تم الانتهاء من استخراج الوجوه وحفظها في: /content/extracted_faces/real
🚀 بدء استخراج عينة الوجوه المزيفة (200 فيديو)...
🔄 جاري معالجة 200 فيديوهات من المجلد...
✅ تم الانتهاء من استخراج الوجوه وحفظها في: /content/extracted_faces/fake
🎉 اكتمل تجهيز البيانات الضخمة بنجاح!


In [ ]:
from google.colab import files
files.download('cloud_deepfake_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

print("📦 جاري ضغط الوجوه الثمينة التي استخرجناها...")
# ضغط المجلد بالكامل
shutil.make_archive("extracted_faces_backup", 'zip', "/content/extracted_faces")

print("✅ تم الضغط بنجاح! جاري التحميل إلى جهازك (قد يأخذ بضع دقائق حسب سرعة الإنترنت)...")
# تحميل الملف لجهازك
files.download("extracted_faces_backup.zip")

📦 جاري ضغط الوجوه الثمينة التي استخرجناها...
✅ تم الضغط بنجاح! جاري التحميل إلى جهازك (قد يأخذ بضع دقائق حسب سرعة الإنترنت)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import os
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
import gc # مكتبة عامل النظافة لتنظيف الرام

# --- 1. تجميع البيانات (النسخة الموفرة للرام 🚀) ---
def load_sequences_optimized(real_folder, fake_folder, sequence_length=10):
    X, Y = [], []
    def process_folder(folder, label):
        faces_dict = {}
        for img_name in os.listdir(folder):
            if not img_name.endswith('.jpg'): continue
            vid_id = img_name.split('_face_')[0]
            if vid_id not in faces_dict:
                faces_dict[vid_id] = []
            faces_dict[vid_id].append(img_name)

        for vid_id, images in faces_dict.items():
            if len(images) >= sequence_length:
                images = sorted(images, key=lambda x: int(x.split('_face_')[1].split('.')[0]))[:sequence_length]
                sequence = []
                for img_name in images:
                    img_path = os.path.join(folder, img_name)
                    img = cv2.imread(img_path)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                    # 💡 السحر هنا: تحويل نوع البيانات إلى float16 يقلل استهلاك الرام بنسبة هائلة!
                    img = img.astype(np.float16) / 255.0
                    sequence.append(img)
                X.append(sequence)
                Y.append(label)

    print("📥 جاري قراءة الوجوه الحقيقية...")
    process_folder(real_folder, 0)
    print("📥 جاري قراءة الوجوه المزيفة...")
    process_folder(fake_folder, 1)

    # 💡 سحر إضافي: تحويل القوائم إلى مصفوفات خفيفة جداً
    return np.array(X, dtype=np.float16), np.array(Y, dtype=np.int8)

X_train, Y_train = load_sequences_optimized('/content/extracted_faces/real', '/content/extracted_faces/fake')

# 🧹 تشغيل عامل النظافة لمسح أي بيانات قديمة عالقة في الرام
gc.collect()

indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
Y_train = Y_train[indices]

print(f"✅ تم تجهيز: {len(X_train)} مقطع فيديو للتدريب! (حجم البيانات في الرام أصبح آمناً جداً)")

# --- 2. بناء العقل ---
print("🧠 جاري بناء معمارية المودل...")
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    TimeDistributed(base_model, input_shape=(10, 224, 224, 3)),
    TimeDistributed(GlobalAveragePooling2D()),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 3. التدريب ---
print("🔥 بدء التدريب العنيف (مع حماية الرام)...")
# 💡 قللنا الدفعة إلى 4 لكي نريح كرت الشاشة تماماً
history = model.fit(X_train, Y_train, epochs=10, batch_size=4, validation_split=0.2)

model.save("cloud_deepfake_model.h5")
print("🎉 مبروك! أصبح المودل ذكياً الآن وتم حفظه بنجاح!")

📥 جاري قراءة الوجوه الحقيقية...
📥 جاري قراءة الوجوه المزيفة...
✅ تم تجهيز: 395 مقطع فيديو للتدريب! (حجم البيانات في الرام أصبح آمناً جداً)
🧠 جاري بناء معمارية المودل...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


🔥 بدء التدريب العنيف (مع حماية الرام)...
Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 123s 496ms/step - accuracy: 0.5190 - loss: 0.7414 - val_accuracy: 0.4937 - val_loss: 0.6950
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 166ms/step - accuracy: 0.5095 - loss: 0.7213 - val_accuracy: 0.5063 - val_loss: 0.7131
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 169ms/step - accuracy: 0.4842 - loss: 0.7292 - val_accuracy: 0.4937 - val_loss: 0.7052
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 169ms/step - accuracy: 0.5348 - loss: 0.7049 - val_accuracy: 0.5063 - val_loss: 0.7078
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 14s 171ms/step - accuracy: 0.5285 - loss: 0.7009 - val_accuracy: 0.5063 - val_loss: 0.6931
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 166ms/step - accuracy: 0.4589 - loss: 0.7080 - val_accuracy: 0.5063 - val_loss: 0.7008
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 20s 165ms/step - accuracy: 0.4747 - loss: 0.7020 - val_accuracy: 0.4937 - val_loss: 0.6936
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 169ms/step - 

🎉 مبروك! أصبح المودل ذكياً الآن وتم حفظه بنجاح!


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential

print("🧠 جاري بناء العقل الجديد المتقدم (Fine-Tuned Model)...")

# 1. إحضار العيون، ولكن هذه المرة سنفتح جزءاً منها!
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True # تفعيل التدريب

# تجميد كل الطبقات القديمة، وترك آخر 20 طبقة فقط لتتعلم ملامح الـ Deepfake
for layer in base_model.layers[:-20]:
    layer.trainable = False

# 2. بناء المعمارية (بشكل أخف قليلاً لمنع الحفظ الأعمى Overfitting)
model = Sequential([
    TimeDistributed(base_model, input_shape=(10, 224, 224, 3)),
    TimeDistributed(GlobalAveragePooling2D()),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 3. سر التعلم الدقيق: تخفيف سرعة التعلم لكي يركز في التفاصيل الصغيرة
custom_adam = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=custom_adam, loss='binary_crossentropy', metrics=['accuracy'])

print("🔥 بدء التدريب العميق (راقب كيف ستكسر الدقة حاجز الـ 50%)...")
# زدنا الجولات إلى 15 لأن التعلم البطيء يحتاج وقتاً أكثر قليلاً
history = model.fit(X_train, Y_train, epochs=15, batch_size=4, validation_split=0.2)

model.save("cloud_deepfake_model.h5")
print("🎉 مبروك! المودل الآن تعلم التفاصيل الدقيقة وتم حفظه!")

🧠 جاري بناء العقل الجديد المتقدم (Fine-Tuned Model)...
🔥 بدء التدريب العميق (راقب كيف ستكسر الدقة حاجز الـ 50%)...
Epoch 1/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 133s 536ms/step - accuracy: 0.4620 - loss: 0.7063 - val_accuracy: 0.5063 - val_loss: 0.6932
Epoch 2/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.4905 - loss: 0.7025 - val_accuracy: 0.4937 - val_loss: 0.6977
Epoch 3/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 191ms/step - accuracy: 0.5032 - loss: 0.6999 - val_accuracy: 0.4937 - val_loss: 0.6965
Epoch 4/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 188ms/step - accuracy: 0.4525 - loss: 0.6985 - val_accuracy: 0.4937 - val_loss: 0.6994
Epoch 5/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 188ms/step - accuracy: 0.4937 - loss: 0.7018 - val_accuracy: 0.4937 - val_loss: 0.6933
Epoch 6/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 187ms/step - accuracy: 0.5000 - loss: 0.7006 - val_accuracy: 0.4937 - val_loss: 0.6937
Epoch 7/15
79/79 ━━━━━━━━━━━━━━━━━━━━ 15s 187ms/step - accuracy: 0.5032 - loss: 0.6978 - val_accuracy: 0.4937 

🎉 مبروك! المودل الآن تعلم التفاصيل الدقيقة وتم حفظه!


In [ ]:
import numpy as np
import os
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint # 💡 سلاح الحفظ التلقائي
import gc

# --- 1. تجميع البيانات بسرعة ---
def load_sequences_fixed(real_folder, fake_folder, sequence_length=10):
    X, Y = [], []
    def process_folder(folder, label):
        faces_dict = {}
        for img_name in os.listdir(folder):
            if not img_name.endswith('.jpg'): continue
            vid_id = img_name.split('_face_')[0]
            if vid_id not in faces_dict:
                faces_dict[vid_id] = []
            faces_dict[vid_id].append(img_name)

        for vid_id, images in faces_dict.items():
            if len(images) >= sequence_length:
                images = sorted(images, key=lambda x: int(x.split('_face_')[1].split('.')[0]))[:sequence_length]
                sequence = []
                for img_name in images:
                    img_path = os.path.join(folder, img_name)
                    img = cv2.imread(img_path)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = img.astype(np.float16)
                    sequence.append(img)
                X.append(sequence)
                Y.append(label)

    print("📥 جاري قراءة الوجوه من الذاكرة...")
    process_folder(real_folder, 0)
    process_folder(fake_folder, 1)

    return np.array(X, dtype=np.float16), np.array(Y, dtype=np.int8)

X_train, Y_train = load_sequences_fixed('/content/extracted_faces/real', '/content/extracted_faces/fake')
gc.collect()

indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
Y_train = Y_train[indices]
print(f"✅ تم تجهيز: {len(X_train)} مقطع فيديو.")

# --- 2. بناء العقل ---
print("🧠 جاري بناء المعمارية...")
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    TimeDistributed(base_model, input_shape=(10, 224, 224, 3)),
    TimeDistributed(GlobalAveragePooling2D()),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 3. التدريب مع الحفظ التلقائي ---
print("🔥 بدء التدريب (مع نظام الحماية ضد التجميد)...")

# 💡 برمجة الحفظ التلقائي: احفظ الملف فقط إذا كانت الدقة أفضل من الجولة السابقة
autosave = ModelCheckpoint("cloud_deepfake_model_autosave.h5",
                           monitor='val_accuracy',
                           save_best_only=True,
                           mode='max',
                           verbose=1)

# أضفنا autosave إلى عملية التدريب
history = model.fit(X_train, Y_train, epochs=10, batch_size=4, validation_split=0.2, callbacks=[autosave])

print("🎉 مبروك! انتهى التدريب بالكامل بنجاح!")

📥 جاري قراءة الوجوه من الذاكرة...
✅ تم تجهيز: 395 مقطع فيديو.
🧠 جاري بناء المعمارية...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


🔥 بدء التدريب (مع نظام الحماية ضد التجميد)...
Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.5610 - loss: 0.7119
Epoch 1: val_accuracy improved from None to 0.50633, saving model to cloud_deepfake_model_autosave.h5



Epoch 1: finished saving model to cloud_deepfake_model_autosave.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 126s 509ms/step - accuracy: 0.5475 - loss: 0.7073 - val_accuracy: 0.5063 - val_loss: 0.6483
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.6293 - loss: 0.6149
Epoch 2: val_accuracy improved from 0.50633 to 0.67089, saving model to cloud_deepfake_model_autosave.h5



Epoch 2: finished saving model to cloud_deepfake_model_autosave.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 166ms/step - accuracy: 0.6108 - loss: 0.6375 - val_accuracy: 0.6709 - val_loss: 0.6268
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.6301 - loss: 0.6263
Epoch 3: val_accuracy improved from 0.67089 to 0.75949, saving model to cloud_deepfake_model_autosave.h5



Epoch 3: finished saving model to cloud_deepfake_model_autosave.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 165ms/step - accuracy: 0.6519 - loss: 0.6025 - val_accuracy: 0.7595 - val_loss: 0.5538
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.6442 - loss: 0.5775
Epoch 4: val_accuracy did not improve from 0.75949
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 160ms/step - accuracy: 0.6772 - loss: 0.5823 - val_accuracy: 0.7089 - val_loss: 0.5701
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.7162 - loss: 0.5377
Epoch 5: val_accuracy did not improve from 0.75949
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 162ms/step - accuracy: 0.6899 - loss: 0.5823 - val_accuracy: 0.5949 - val_loss: 0.6208
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.7328 - loss: 0.5291
Epoch 6: val_accuracy did not improve from 0.75949
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 163ms/step - accuracy: 0.7278 - loss: 0.5206 - val_accuracy: 0.6709 - val_loss: 0.6040
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/st


Epoch 8: finished saving model to cloud_deepfake_model_autosave.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 170ms/step - accuracy: 0.7816 - loss: 0.4712 - val_accuracy: 0.8481 - val_loss: 0.4610
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.8082 - loss: 0.4365
Epoch 9: val_accuracy did not improve from 0.84810
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 167ms/step - accuracy: 0.8354 - loss: 0.3984 - val_accuracy: 0.6076 - val_loss: 0.7970
Epoch 10/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.8171 - loss: 0.3762
Epoch 10: val_accuracy did not improve from 0.84810
79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 168ms/step - accuracy: 0.8259 - loss: 0.3859 - val_accuracy: 0.7089 - val_loss: 0.5919
🎉 مبروك! انتهى التدريب بالكامل بنجاح!


In [ ]:
from google.colab import files
files.download("cloud_deepfake_model_autosave.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>